<a href="https://colab.research.google.com/github/naheejeong/cs182-final-project/blob/main/01_llama_vulnerability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# # hardware check - GPU & memory assigned
# !nvidia-smi

In [25]:
# Connect Google Drive to the Colab machine so results are saved to Drive.
from google.colab import drive
drive.mount('/content/drive')

# Setup commands (! = system command, % = notebook command)
%cd /content

# Download the InjecAgent benchmark (attack cases and evaluation code) from GitHub.
# --> this makes a new folder named "InjecAgent" in /content.
!git clone https://github.com/uiuc-kang-lab/InjecAgent.git
%cd /content/InjecAgent

# Install the 6 required packages (4 from requirements.txt + together and nltk).
!pip install -q -r requirements.txt together nltk

# Create a results folder in Google Drive (My Drive/cs182/results).
!mkdir -p /content/drive/MyDrive/cs182/results

# Create shortcut: InjecAgent/results -> My Drive/cs182/results, so outputs are saved to Drive.
!ln -sfn /content/drive/MyDrive/cs182/results results

# Provide environment variables needed by the InjecAgent code.
import os
os.environ["PYTHONPATH"] = "/content/InjecAgent"
os.environ["OPENAI_API_KEY"] = "unused" # placeholder; we never use!

!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'InjecAgent' already exists and is not an empty directory.
/content/InjecAgent
asset  data  LICENCE  README.md  requirements.txt  results  src


In [26]:
# Log in to HF using the token stored in Secrets. Token name: HF_TOKEN, username: naheejeong
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download
login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()['name'])

# Testing Llama access - download one tiny file from the model's page.
path = hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json")
print(path)
print(open(path).read())    # printing the config file

Logged in as: naheejeong
/root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/config.json
{
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "low_freq_factor": 1.0,
    "high_freq_factor": 4.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.42.3",
  "use_cache": true,
  "vocab_si

In [27]:
# A little background in InjecAgent:
# InjecAgent doesn’t care which model it tests, but it requires that each model is wrapped in a class with three functions.
# Below code shows the template its authors wrote:
!head -n 12 /content/InjecAgent/src/models.py

import os
from src.utils import get_response_text
import time
class BaseModel:
    def __init__(self):
        self.model = None

    def prepare_input(self, sys_prompt,  user_prompt_filled):
        raise NotImplementedError("This method should be overridden by subclasses.")

    def call_model(self, model_input):
        raise NotImplementedError("This method should be overridden by subclasses.")


In [28]:
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)   # downloads Llama's tokenizer

# tok("some text") returns:
# {
#  "input_ids":      [128000, 2028, 374, ...],
#  "attention_mask": [1, 1, 1, ...]
# }
ids = tok("This is a test input Meow :3")["input_ids"] # --> E.g., [128000, 2028, 374, ...]. Turning text into ids.

print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

[128000, 2028, 374, 264, 1296, 1988, 2206, 363, 551, 18]
['<|begin_of_text|>', 'This', 'Ġis', 'Ġa', 'Ġtest', 'Ġinput', 'ĠMe', 'ow', 'Ġ:', '3']
<|begin_of_text|>This is a test input Meow :3


In [29]:
from torch import bfloat16
from transformers import AutoModelForCausalLM
from transformers import __version__ as transformers_version

print("transformers version:", transformers_version)

# Download Llama's weights (~16 GB) and load them onto the GPU at half precision
# FYI - Half precision just means each parameter is stored in 16 bits instead of 32,
# so the memory is halved (~16 GB instead of ~32 GB).
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=bfloat16,   # Meta published this model's weights in bfloat16 (half precision)
    device_map="auto",    # put the model on GPU automatically
)

!nvidia-smi

transformers version: 5.22.1


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

/bin/bash: line 1: nvidia-smi: command not found
